# Tìm siêu tham số và đánh giá: GAT-GRU

1. **Tìm siêu tham số** bằng Optuna (TPE sampler + Median pruner) trên tập feature đã chọn. Mục tiêu là **val MAE** (thang đã chuẩn hóa, trung bình 2 target). Ngân sách (`ft.BUDGET`) **giống hệt** cho LSTM và GAT-GRU: cùng số trial, cùng số epoch tối đa, cùng early stopping, cùng seed của sampler.
2. **Huấn luyện lại** cấu hình tốt nhất với 5 seed, rồi đánh giá **một lần** trên test ở đơn vị gốc (MAE/RMSE), báo cáo trung bình ± độ lệch chuẩn qua các seed.
3. **Lưu** `gatgru_predictions_test.npz` theo quy ước của `forecast_metrics.py`, cho notebook so sánh.

Không gian tìm kiếm (horizon h = 6 cố định):

| Siêu tham số | Giá trị |
|---|---|
| window w | {6, 12, 18} bước |
| dropout | [0; 0,3] |
| learning rate | [1e-4; 3e-3] (log) |
| batch size | {64, 128, 256} |
| GAT hidden × heads | {16, 32, 64} × {1, 2, 4} |
| số lớp GAT | {1, 2} |
| GRU hidden | {32, 64, 128} |
| hướng cạnh | caller→callee, callee→caller, cả hai |
| dùng feature cạnh | có / không |

**Chọn bộ dataset** bằng biến `DATASET` (đặt **giống nhau** ở mọi notebook của một lần thí nghiệm):

| `DATASET` | Nguồn | Kịch bản | Split | Thư mục đầu ra |
|---|---|---|---|---|
| `'load-testing'` | `load-testing/results/` (Locust, mô hình đóng) | normal, spike, bursty | `modeling-common/splits.json` | `processed/` |
| `'math'` | `math-load-testing/results/` (mô hình toán, luồng đến mở) | nhpp, mmpp, pareto, onoff | `modeling-common/splits_math.json` | `processed_math/` |

Hai bộ có split và thư mục kết quả riêng nên không ảnh hưởng nhau.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

COMMON = (Path('..') / 'modeling-common').resolve()   # code dùng chung cho GAT-GRU và LSTM
sys.path.insert(0, str(COMMON))
import forecast_data as fd

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 180)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

import forecast_metrics as fm
import forecast_torch as ft
import torch

In [ ]:
DATASET = 'load-testing'          # 'load-testing' | 'math' - giống notebook tiền xử lý
TAG = '_math' if DATASET == 'math' else ''
PROCESSED = Path(f'processed{TAG}') if (Path(f'processed{TAG}') / 'runs_raw.npz').exists() else Path(f'processed{TAG}_demo')
DEMO = PROCESSED.name.endswith('_demo')
MODEL = 'gatgru'
B = ft.DEMO_BUDGET if DEMO else ft.BUDGET
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
runs, meta = fd.load_runs_npz(PROCESSED / 'runs_raw.npz')
splits = meta['splits']
edges = [tuple(e) for e in meta['edges']]
sel_path = PROCESSED / 'selected_features.json'
if sel_path.exists():
    sel = json.loads(sel_path.read_text())
else:
    print('⚠️ Chưa chạy feature_selection: dùng mọi feature ứng viên')
    sel = {'node': fd.NF, 'edge': fd.EF if MODEL == 'gatgru' else []}
data = ft.Data(runs, splits, sel['node'], sel['edge'] if MODEL == 'gatgru' else None)
print('device:', DEVICE, '| budget:', B)
print('node features:', data.node_feats)
print('edge features:', data.edge_feats)

## 1. Optuna

In [ ]:
study = ft.run_optuna(MODEL, data, edges, n_trials=B['n_trials'], seed=B['search_seed'], device=DEVICE,
                      max_epochs=B.get('max_epochs'))
trials = study.trials_dataframe(attrs=('number', 'value', 'params', 'state'))
print('best val MAE (scaled):', round(study.best_value, 4))
print('best params:', study.best_params)
trials.sort_values('value').head(10)

In [ ]:
done = trials[trials.state == 'COMPLETE'].sort_values('number')
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(done.number, done.value, 'o', alpha=0.5, label='trial')
axes[0].plot(done.number, done.value.cummin(), 'r-', label='tốt nhất đến hiện tại')
axes[0].set_xlabel('trial')
axes[0].set_ylabel('val MAE (scaled)')
axes[0].legend()
axes[0].set_title('Lịch sử tối ưu')
try:
    import optuna
    imp = optuna.importance.get_param_importances(study)
    axes[1].barh(list(imp)[::-1], list(imp.values())[::-1])
    axes[1].set_title('Mức ảnh hưởng của siêu tham số (fANOVA)')
except Exception as exc:
    axes[1].text(0.1, 0.5, f'không tính được: {exc}', wrap=True)
fig.tight_layout()
plt.show()

## 2. Huấn luyện lại với nhiều seed và đánh giá trên test

In [ ]:
cfg = ft.study_config(study, MODEL, bool(data.edge_feats))
if B.get('max_epochs'):
    cfg['max_epochs'] = B['max_epochs']
preds, test_w, models, meta_m = ft.final_runs(MODEL, cfg, data, edges, seeds=B['final_seeds'], device=DEVICE)
meta_m['demo'] = DEMO
fm.save_predictions(PROCESSED / f'{MODEL}_predictions_test.npz', preds, test_w['Y_raw'], test_w['Y_last'],
                    test_w['info'], meta_m)
ft.save_model(PROCESSED / f'{MODEL}_model.pt', models[0], meta_m)
(PROCESSED / f'{MODEL}_best_config.json').write_text(json.dumps(cfg, indent=1))
print(f"params: {meta_m['n_params']:,} | inference 1 window: {meta_m['inference_ms']:.2f} ms | test windows: {len(test_w['X'])}")

d = fm.load_predictions(PROCESSED / f'{MODEL}_predictions_test.npz')
summary, _, _ = fm.seed_summary(d, fd.TARGETS, MODEL.upper())
pers = fm.error_table(fm.persistence(d), d['Y_true'], fd.TARGETS).round(4).astype(str)
pers.insert(0, 'model', 'persistence')
pd.concat([summary, pers])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for k, t in enumerate(fd.TARGETS):
    h = np.arange(1, fd.HORIZON + 1) * fd.STEP_SEC
    per_seed = np.stack([fm.by_horizon(d['Y_pred'][s], d['Y_true'], k) for s in range(len(d['Y_pred']))])
    axes[k].errorbar(h, per_seed.mean(0), yerr=per_seed.std(0), marker='o', capsize=3, label=MODEL.upper())
    axes[k].plot(h, fm.by_horizon(fm.persistence(d), d['Y_true'], k), 's--', label='persistence')
    axes[k].set_xlabel('dự báo trước (giây)')
    axes[k].set_ylabel(f'MAE {t}')
    axes[k].legend()
fig.suptitle('Sai số theo khoảng dự báo (test)')
fig.tight_layout()
plt.show()

In [ ]:
# Ví dụ dự báo frontend rps_in trên run test đầu tiên (seed 0, bước dự báo t+1 và t+6)
f = fd.SERVICES.index('frontend')
key = d['info'][0, 0]
m = d['info'][:, 0] == key
t = d['info'][m, 2].astype(int)
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(t, d['Y_true'][m, 0, f, 0], 'k-', lw=1.5, label='thực tế')
ax.plot(t, d['Y_pred'][0][m, 0, f, 0], label='dự báo t+1')
ax.plot(t + fd.HORIZON - 1, d['Y_pred'][0][m, -1, f, 0], label=f't+{fd.HORIZON}', alpha=0.8)
ax.set_title(f'{key}: frontend rps_in')
ax.set_xlabel('bước (10s)')
ax.legend()
plt.show()

## 3. Attention đã học
Trung bình hệ số attention $\alpha_{ij}$ của lớp GAT cuối trên tập test: service *i* (hàng) dựa vào service *j* (cột) bao nhiêu. Đây là trọng số phụ thuộc động mà đề cương muốn học từ dữ liệu trace.

In [ ]:
model = models[0].eval()
w = test_w
with torch.no_grad():
    E = torch.from_numpy(w['E'][:256]).to(DEVICE) if 'E' in w and cfg.get('use_edge_features') else None
    model(torch.from_numpy(w['X'][:256]).to(DEVICE), E)
att = model.last_attention.mean(dim=(0, 1, 4)).cpu().numpy()     # [N_dst, N_src]
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(att, cmap='viridis')
ax.set_xticks(range(len(fd.SERVICES)), fd.SERVICES, rotation=75, fontsize=8)
ax.set_yticks(range(len(fd.SERVICES)), fd.SERVICES, fontsize=8)
ax.set_xlabel('nguồn j')
ax.set_ylabel('đích i')
ax.set_title(f"Attention trung bình (edge_direction = {cfg['edge_direction']})")
fig.colorbar(im)
plt.show()